In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

path = os.path.join("..", "data", "loan_cleaned.csv")

try:
    df=pd.read_csv(path)
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    print(f"Display first few rows")
    display(df.head())

except FileNotFoundError:
    print(f"Error: Could not find the file at '{csv_path}'.")    

Dataset Shape: 148670 rows, 34 columns

Display first few rows


,loan_id,year,loan_limit,gender,approved_in_advance,loan_type,loan_purpose,credit_worthiness,open_credit,business_or_commercial,...,credit_type,credit_score,co_applicant_credit_type,age,submission_of_application,loan_to_value_ratio,region,security_type,status,debt_to_income_ratio
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


DROP columns

In [2]:
columns_to_drop = ['loan_id', 'year']
df = df.drop(columns=columns_to_drop)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(f"Display first few rows")
display(df.head().T)

Dataset Shape: 148670 rows, 32 columns

Display first few rows


,0,1,2,3,4
loan_limit,cf,cf,cf,cf,cf
gender,Sex Not Available,Male,Male,Male,Joint
approved_in_advance,nopre,nopre,pre,nopre,pre
loan_type,type1,type2,type1,type1,type1
loan_purpose,p1,p1,p1,p4,p1
credit_worthiness,l1,l1,l1,l1,l1
open_credit,nopc,nopc,nopc,nopc,nopc
business_or_commercial,nob/c,b/c,nob/c,nob/c,nob/c
loan_amount,116500,206500,406500,456500,696500
rate_of_interest,NaN,NaN,4.56,4.25,4.0


In [3]:
df['upfront_charges_missing'] = df['upfront_charges'].isnull().astype(int)

# Test against approved_in_advance
display(pd.crosstab(df['upfront_charges_missing'], df['approved_in_advance'], normalize='index'))

approved_in_advance,nopre,pre
upfront_charges_missing,,
0,0.835068,0.164932
1,0.866291,0.133709


## Impute upfront_charges with median


In [4]:
# Check Mean interest rate for Non-Default vs Default
print(df.groupby('status')['rate_of_interest'].mean())


status
0    4.044931
1    4.350500
Name: rate_of_interest, dtype: float64


**rate_of_interest** — Dropped. The interest rate is set by the lender post-approval. 
Not available at application time. Confirmed leakage: defaulters show 
measurably higher rates (4.35% vs 4.04%), meaning the model would learn 
a real but unusable signal.

**interest_rate_spread** — Dropped. Derived from rate_of_interest. 
The same leakage reasoning applies.

In [5]:
# Drop leaky features that are determined post-approval
df = df.drop(columns=['rate_of_interest', 'interest_rate_spread'])
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

Dataset Shape: 148670 rows, 31 columns



In [6]:
df['dtir_missing'] = df['debt_to_income_ratio'].isnull().astype(int)

# Does income missingness predict dtir missingness?
print(df['dtir_missing'].corr(df['income'].isnull().astype(int)))

# Is income lower when dtir is missing?
print(df.groupby('dtir_missing')['income'].mean())

# Does credit score differ?
print(df.groupby('dtir_missing')['credit_score'].mean())

0.5735713723835258
dtir_missing
0    7013.189434
1    6496.494927
Name: income, dtype: float64
dtir_missing
0    699.797823
1    699.744082
Name: credit_score, dtype: float64


In [7]:
both_missing = df['debt_to_income_ratio'].isnull() & df['income'].isnull()
print(f"Rows where both dtir and income are missing: {both_missing.sum()}")
print(f"As % of dataset: {both_missing.sum()/len(df)*100:.2f}%")

Rows where both dtir and income are missing: 9040
As % of dataset: 6.08%


In [8]:
# Create flags BEFORE imputing — once you fill nulls, the information is gone
df['income_missing'] = df['income'].isnull().astype(int)
df['dtir_missing'] = df['debt_to_income_ratio'].isnull().astype(int)

# Now check if these flags correlate with default
print(df.groupby('income_missing')['status'].mean())
print(df.groupby('dtir_missing')['status'].mean())

income_missing
0    0.253727
1    0.135410
Name: status, dtype: float64
dtir_missing
0    0.163221
1    0.676174
Name: status, dtype: float64


## Missingness Indicator Features

- **income_was_missing**: Income missing correlates with LOWER default (13.5% vs 25.4%).
  Likely explanation: asset-rich borrowers don't report income.
  
- **dtir_was_missing**: DTI missing correlates with MUCH HIGHER default (67.6% vs 16.3%).
  Strongest missingness signal in dataset. Retained as feature.

In [9]:
df.head().T

,0,1,2,3,4
loan_limit,cf,cf,cf,cf,cf
gender,Sex Not Available,Male,Male,Male,Joint
approved_in_advance,nopre,nopre,pre,nopre,pre
loan_type,type1,type2,type1,type1,type1
loan_purpose,p1,p1,p1,p4,p1
credit_worthiness,l1,l1,l1,l1,l1
open_credit,nopc,nopc,nopc,nopc,nopc
business_or_commercial,nob/c,b/c,nob/c,nob/c,nob/c
loan_amount,116500,206500,406500,456500,696500
upfront_charges,NaN,NaN,595.0,NaN,0.0


## Impute DTIR | Upfront Charges | Income with median

In [10]:
# 1. Calculate the medians
upfront_median = df['upfront_charges'].median()
dtir_median = df['debt_to_income_ratio'].median()
income_median = df['income'].median()

#2. Impute columns
df['upfront_charges'] = df['upfront_charges'].fillna(upfront_median)
df['debt_to_income_ratio'] = df['upfront_charges'].fillna(dtir_median)
df['income'] = df['upfront_charges'].fillna(income_median)

print(df[['upfront_charges','debt_to_income_ratio','income']].isna().sum())

upfront_charges         0
debt_to_income_ratio    0
income                  0
dtype: int64


In [11]:
#check the count when both property value and LTV is missing 
both_missing = df['property_value'].isnull() & df['loan_to_value_ratio'].isnull()
print(both_missing.sum())

15098


In [12]:
#create a feature flag to property_value missing
#This flag creates a leaky feature compared with 
df['propval_missing']=df['property_value'].isnull().astype(int)
print(df.groupby('propval_missing')['status'].mean())

propval_missing
0    0.161284
1    0.999868
Name: status, dtype: float64


In [13]:
propval_median = df['property_value'].median()
loanval_median = df['loan_to_value_ratio'].median()

df['property_value']= df['property_value'].fillna(propval_median)
df['loan_to_value_ratio']= df['loan_to_value_ratio'].fillna(loanval_median)
print(df[['property_value','loan_to_value_ratio']].isnull().sum())

#drop propval_missing column as it's creating a leaky feature
df = df.drop(columns=['propval_missing'])

property_value         0
loan_to_value_ratio    0
dtype: int64


In [14]:
# 1. Categorical imputation with Mode
categorical_cols_to_impute = [
    'loan_limit', 
    'approved_in_advance', 
    'submission_of_application', 
    'age', 
    'loan_purpose', 
    'negative_amortization'
]

for col in categorical_cols_to_impute:
    # .mode()[0] picks the most frequent value
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

# 2. Numerical imputation with Median
term_median = df['term'].median()
df['term'] = df['term'].fillna(term_median)

# 3. Verify that these columns have 0 missing values now
print("Remaining missing values for imputed columns:")
print(df[categorical_cols_to_impute+['term']].isnull().sum())

Remaining missing values for imputed columns:
loan_limit                   0
approved_in_advance          0
submission_of_application    0
age                          0
loan_purpose                 0
negative_amortization        0
term                         0
dtype: int64


In [15]:
print(df.isnull().sum())

loan_limit                   0
gender                       0
approved_in_advance          0
loan_type                    0
loan_purpose                 0
credit_worthiness            0
open_credit                  0
business_or_commercial       0
loan_amount                  0
upfront_charges              0
term                         0
negative_amortization        0
interest_only                0
lump_sum_payment             0
property_value               0
construction_type            0
occupancy_type               0
secured_by                   0
total_units                  0
income                       0
credit_type                  0
credit_score                 0
co_applicant_credit_type     0
age                          0
submission_of_application    0
loan_to_value_ratio          0
region                       0
security_type                0
status                       0
debt_to_income_ratio         0
upfront_charges_missing      0
dtir_missing                 0
income_m

In [16]:
print(df.shape)
print(df.columns.tolist())

(148670, 33)
['loan_limit', 'gender', 'approved_in_advance', 'loan_type', 'loan_purpose', 'credit_worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'upfront_charges', 'term', 'negative_amortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'secured_by', 'total_units', 'income', 'credit_type', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'security_type', 'status', 'debt_to_income_ratio', 'upfront_charges_missing', 'dtir_missing', 'income_missing']


In [17]:
# Check how negative amortization categories relate to default status
print(df.groupby('negative_amortization')['status'].mean())

# Also check if it's heavily correlated with missing interest rates
print(pd.crosstab(df['negative_amortization'], df['income_missing']))

negative_amortization
neg_amm    0.445965
not_neg    0.223841
Name: status, dtype: float64
income_missing              0     1
negative_amortization              
neg_amm                 14680   449
not_neg                124840  8701


In [18]:
for col in ['security_type', 'secured_by', 'construction_type', 'open_credit','lump_sum_payment','credit_worthiness','interest_only','total_units']:
    print(col)
    print(df.groupby(col)['status'].mean())
    print()

security_type
security_type
Indriect    1.000000
direct      0.246278
Name: status, dtype: float64

secured_by
secured_by
home    0.246278
land    1.000000
Name: status, dtype: float64

construction_type
construction_type
mh    1.000000
sb    0.246278
Name: status, dtype: float64

open_credit
open_credit
nopc    0.246709
opc     0.176259
Name: status, dtype: float64

lump_sum_payment
lump_sum_payment
lpsm        0.776596
not_lpsm    0.234097
Name: status, dtype: float64

credit_worthiness
credit_worthiness
l1    0.243277
l2    0.317736
Name: status, dtype: float64

interest_only
interest_only
int_only    0.273136
not_int     0.245105
Name: status, dtype: float64

total_units
total_units
1U    0.244969
2U    0.345295
3U    0.384224
4U    0.296875
Name: status, dtype: float64



In [19]:
weird = df[df['secured_by'] == 'land']
print(weird.shape)
print(weird['status'].value_counts())
print(weird[['security_type', 'construction_type']].value_counts())

(33, 33)
status
1    33
Name: count, dtype: int64
security_type  construction_type
Indriect       mh                   33
Name: count, dtype: int64


In [20]:
print(df['total_units'].value_counts())

total_units
1U    146480
2U      1477
3U       393
4U       320
Name: count, dtype: int64


In [21]:
print(df['interest_only'].value_counts())

interest_only
not_int     141560
int_only      7110
Name: count, dtype: int64


In [22]:
df = df.drop(columns=['security_type', 'secured_by', 'construction_type', 
                       'open_credit', 'credit_worthiness', 'interest_only','upfront_charges_missing'])
print(df.shape)

(148670, 26)


In [23]:
print(df.columns.tolist())

['loan_limit', 'gender', 'approved_in_advance', 'loan_type', 'loan_purpose', 'business_or_commercial', 'loan_amount', 'upfront_charges', 'term', 'negative_amortization', 'lump_sum_payment', 'property_value', 'occupancy_type', 'total_units', 'income', 'credit_type', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'status', 'debt_to_income_ratio', 'dtir_missing', 'income_missing']


In [24]:
for col in ['loan_limit', 'occupancy_type', 'negative_amortization', 
            'business_or_commercial', 'approved_in_advance']:
    print(f"\n{col}")
    print(df[col].value_counts())
    print(df.groupby(col)['status'].mean())


loan_limit
loan_limit
cf     138692
ncf      9978
Name: count, dtype: int64
loan_limit
cf     0.240281
ncf    0.332131
Name: status, dtype: float64

occupancy_type
occupancy_type
pr    138201
ir      7340
sr      3129
Name: count, dtype: int64
occupancy_type
ir    0.299864
pr    0.243045
sr    0.271333
Name: status, dtype: float64

negative_amortization
negative_amortization
not_neg    133541
neg_amm     15129
Name: count, dtype: int64
negative_amortization
neg_amm    0.445965
not_neg    0.223841
Name: status, dtype: float64

business_or_commercial
business_or_commercial
nob/c    127908
b/c       20762
Name: count, dtype: int64
business_or_commercial
b/c      0.345439
nob/c    0.230377
Name: status, dtype: float64

approved_in_advance
approved_in_advance
nopre    125529
pre       23141
Name: count, dtype: int64
approved_in_advance
nopre    0.253360
pre      0.208937
Name: status, dtype: float64


In [25]:
df = df.drop(columns=['loan_limit', 'occupancy_type', 'approved_in_advance'])
print(df.shape)

(148670, 23)


In [26]:
print(df.columns.tolist())


['gender', 'loan_type', 'loan_purpose', 'business_or_commercial', 'loan_amount', 'upfront_charges', 'term', 'negative_amortization', 'lump_sum_payment', 'property_value', 'total_units', 'income', 'credit_type', 'credit_score', 'co_applicant_credit_type', 'age', 'submission_of_application', 'loan_to_value_ratio', 'region', 'status', 'debt_to_income_ratio', 'dtir_missing', 'income_missing']


### 1. Zero Variance / Uniformity

Dropped because they were heavily dominated by a single category:

* **`security_type`** – Over **99.9%** was `direct`.
* **`open_credit`** – Almost entirely `nopc`.
* **`credit_worthiness`** – Overwhelmingly skewed toward `l1`.

### 2. Extreme Imbalance / Small Sample Biases

Dropped because rare subcategories had a literal **100% default rate** ($status = 1.0$), which risks severe model overfitting:

* **`secured_by`** – The category `land` had a 100% default rate.
* **`construction_type`** – The category `mh` had a 100% default rate.
* **`interest_only`** – The category `sr_no` had a 100% default rate.

### 3. Weak Predictive Separation

Dropped because default rates were too similar across categories to provide useful predictive power:

* **`loan_limit`** – Small contrast between `cf` (~24%) and `ncf` (~33%).
* **`occupancy_type`** – All categories (`pr`, `ir`, `sr`) clustered tightly between **24% and 30%**.
* **`approved_in_advance`** – Negligible difference between `nopre` (~25.3%) and `pre` (~20.9%).

### 4. Technical Tracking

* **`upfront_charges_missing`** – Temporary binary helper column from the missing data imputation step.